In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
df = pd.read_csv("finalproject_data.csv").copy()
df["duration_ms"] = pd.to_numeric(df["duration_ms"], errors="coerce")
df["duration_min"] = df["duration_ms"] / 60000
features = [
    "danceability","energy","valence","tempo",
    "acousticness","instrumentalness",
    "liveness","speechiness","loudness","duration_min"
]
df = df.dropna(subset=features + ["name","artists"]).reset_index(drop=True)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features])

/var/folders/z8/6ldbxn9140g5h_3x3qprykt40000gn/T/ipykernel_40223/4192875654.py:5: DtypeWarning: Columns (3,4,5,6,7,8,9,10,11,12,13,14,17,18,19,29,33) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("finalproject_data.csv").copy()


In [3]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features])

In [4]:
liked_songs = [("Let Me Love You (Until You Learn To Love Yourself)", "Ne-Yo"),
              ("Photograph","The Verve Pipe"),("A Perfect Sonnet","Bright Eyes"),("Someone You Loved","Lewis Capaldi"),("All of Me","Willie Nelson"),
              ("Stay With Me","Rod Stewart"),("Let Her Go","Jasmine Thompson"),("Blinding Lights", "The Weeknd"),("Levitating", "Dua Lipa"),
               ("Don't Start Now", "Dua Lipa")]

In [5]:
def build_user_profile(liked_songs):
    liked_indices = []

    for title, artist in liked_songs:
        matches = df[
            (df["name"].str.lower() == title.lower()) &
            (df["artists"].str.contains(artist, case=False, na=False))
        ]
        if matches.empty:
            print(f"❌ Not found: {title} - {artist}")
            continue
        idx = matches.sort_values("popularity", ascending=False).index[0]
        liked_indices.append(idx)
    if len(liked_indices) == 0:
        raise ValueError("No liked songs found")
    user_profile = np.mean(X_scaled[liked_indices], axis=0)
    return user_profile, liked_indices
user_profile, liked_indices = build_user_profile(liked_songs)

In [20]:
profile_df = pd.DataFrame([user_profile], columns=features)
print(profile_df.T.sort_values(by=0, ascending=False))

                         0
loudness          0.746522
energy            0.619574
tempo             0.403841
duration_min      0.028955
liveness          0.010076
danceability     -0.009784
speechiness      -0.062387
valence          -0.078197
instrumentalness -0.465729
acousticness     -0.547726


In [6]:
custom_filter = {
    "energy_min": 0.55,
    "loudness_min": -8,      
    "acousticness_max": 0.35
}

In [7]:
def recommend(liked_songs, custom_filter=None, top_n=10):
    user_profile, liked_indices = build_user_profile(liked_songs)
    candidate_df = df.copy()   
    if custom_filter:
        for key, value in custom_filter.items():
            if key.endswith("_min"):
                col = key.replace("_min","")
                candidate_df = candidate_df[candidate_df[col] >= value]
            elif key.endswith("_max"):
                col = key.replace("_max","")
                candidate_df = candidate_df[candidate_df[col] <= value]
    candidate_idx = candidate_df.index.tolist()
    candidate_idx = [i for i in candidate_idx if i not in liked_indices]
    sims = cosine_similarity([user_profile], X_scaled[candidate_idx])[0]
    ranked = sorted(zip(candidate_idx, sims), key=lambda x: x[1], reverse=True)
    top_indices = [x[0] for x in ranked[:top_n]]
    return df.loc[top_indices, ["name","artists","year","popularity"]]

In [8]:
for col in features:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df = df.dropna(subset=features + ["name","artists"]).reset_index(drop=True)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features])
recommend(liked_songs, custom_filter=custom_filter)

,name,artists,year,popularity
6861,Don't Give up on Me,['Jason Aldean'],2009.0,44.0
95256,Middle of a Memory,['Cole Swindell'],2016.0,65.0
122764,Through the Dark,['One Direction'],2013,64
148596,The Other Side (feat. CeeLo Green and B.o.B),"['Bruno Mars', 'B.o.B', 'CeeLo Green']",2010,52
163385,The Morning Papers,['Prince'],1992,40
60850,The Beers,['The Front Bottoms'],2011.0,42.0
149361,Mysteries of the World,['Walker McGuire'],2018,64
104682,Yours,['Russell Dickerson'],2017.0,67.0
7066,Guns,['Justin Moore'],2011.0,46.0
113375,Granddaddy's Gun,['Aaron Lewis'],2012.0,56.0
